# 02 - Clean Weather (Silver)

From `bronze.weather_raw`:
1. Strip string columns
2. Round numerics
3. Write `silver.weather_clean`
4. Report provinces / quarters / ENSO (no hardcoded expectations)

## 1. Configuration

In [0]:
from pyspark.sql import functions as F

from configs.project_config import BRONZE_TABLES, SILVER_TABLES, VALID_QUARTERS

bronze_table = BRONZE_TABLES['weather_raw']
silver_table = SILVER_TABLES['weather_clean']

print(f'Source : {bronze_table}')
print(f'Target : {silver_table}')

## 2. Load Bronze

In [0]:
raw = spark.table(bronze_table)
print(f'Bronze rows: {raw.count()}')

## 3. Strip strings

In [0]:
df = (
    raw
    .withColumn('province', F.trim(F.col('province')))
    .withColumn('quarter', F.trim(F.col('quarter')))
    .withColumn('year_quarter', F.trim(F.col('year_quarter')))
    .withColumn('enso_phase', F.trim(F.col('enso_phase')))
)

## 4. Round numerics + cast

In [0]:
round_2 = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_sum',
    'shortwave_radiation_sum', 'reference_evapotranspiration_mm', 'wind_speed_10m_max', 'wind_gusts_10m_max',
    'rain_normal', 'temp_normal', 'rainfall_deviation_pct'
]
round_3 = ['temperature_anomaly_c', 'oni_index']
round_0 = ['sunshine_duration']

for c in round_2:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 2))
for c in round_3:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 3))
for c in round_0:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 0))

if 'precipitation_hours' in df.columns:
    df = df.withColumn('precipitation_hours', F.col('precipitation_hours').cast('int'))

## 5. Write Silver table

In [0]:
(
    df.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)
print(f'Wrote: {silver_table}')

## 6. Validation

In [0]:
result = spark.table(silver_table)

print(f'Rows : {result.count()}')
print(f'Cols : {len(result.columns)}')

checks = {
    'null province': result.filter(F.col('province').isNull()).count(),
    'null year': result.filter(F.col('year').isNull()).count(),
    'null quarter': result.filter(F.col('quarter').isNull()).count(),
    'invalid quarter': result.filter(~F.col('quarter').isin(VALID_QUARTERS)).count()
}

print('DQ checks (expect 0):')
for name, cnt in checks.items():
    print(f'  {name}: {cnt}')

print()
print('Provinces:')
result.select('province').distinct().orderBy('province').show(60, truncate=False)
print('Quarters:')
result.groupBy('quarter').count().orderBy('quarter').show()
print('ENSO phases:')
result.groupBy('enso_phase').count().orderBy('enso_phase').show()

print()
print('Next: 03_enrich_crop_weather')